# Similaridade Semântica com Word2Vec
Este notebook treina um modelo Word2Vec (Skip-gram) a partir de dois documentos (um arquivo Word e um PDF) e usa os vetores para comparar textos.

Fluxo geral:
1. Carregamento dos documentos de origem e pré-processamento.
2. Treinamento do Word2Vec com sg=1 (Skip-gram).
3. Leitura de dois arquivos CSV e cálculo da similaridade (0.0 a 1.0) entre cada combinação de textos.

In [ ]:
# %pip install -q gensim python-docx pdfplumber pandas numpy

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires pydantic, which is not installed.
google-generativeai 0.8.5 requires tqdm, which is not installed.
keras 3.7.0 requires h5py, which is not installed.
keras 3.7.0 requires rich, which is not installed.
langchain-core 0.3.76 requires jsonpatch<2.0,>=1.33, which is not installed.
langchain-core 0.3.76 requires pydantic>=2.7.4, which is not installed.
langchain-core 0.3.76 requires PyYAML>=5.3, which is not installed.
langchain-core 0.3.76 requires tenacity!=8.4.0,<10.0.0,>=8.1.0, which is not installed.
langchain-google-genai 2.0.10 requires pydantic<3,>=2, which is not installed.
tensorboard 2.18.0 requires markdown>=2.6.8, which is not installed.
tensorboard 2.18.0 requires werkzeug>=1.0.1, which is not installed.
tensorflow-intel 2.18.0 requires h5py>=3.11.0, which is not install

In [2]:
import os
import re
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from docx import Document
import pdfplumber

## Configurações de caminhos e colunas
Ajuste os caminhos abaixo para apontar para os seus arquivos reais. Os CSVs devem conter ao menos duas colunas; indique quais serão usadas para comparação.

In [13]:
DATA_DIR = Path('C:\\')
DOCX_FILE = DATA_DIR / 'Users\\anton\\OneDrive\\Documentos\\Projetos\\Fatec\\Comparador de Ementas\\Projeto Pedagógico do Curso Superior de Tecnologia em ADS - Fatec Zona Leste 2022.10.10.docx'
PDF_FILE = DATA_DIR / 'Users\\anton\\OneDrive\\Documentos\\Projetos\\Fatec\\Comparador de Ementas\\PPC_CST Análise e Desenvolvimento de Sistemas.pdf'
CSV_A_PATH = DATA_DIR /  'Users\\anton\\OneDrive\\Documentos\\Projetos\\Fatec\\Comparador de Ementas\\disciplinas-curso-antigo.csv'
CSV_B_PATH = DATA_DIR / 'Users\\anton\\OneDrive\\Documentos\\Projetos\\Fatec\\Comparador de Ementas\\disciplinas-curso-novo.csv'


COLUMN_DF_A = 'syllabus'  # coluna do primeiro dataframe
COLUMN_DF_B = 'syllabus'  # coluna do segundo dataframe
WORD2VEC_VECTOR_SIZE = 100
WORD2VEC_EPOCHS = 20

## Funções utilitárias
Responsáveis por ler arquivos, limpar texto, gerar sentenças e treinar o modelo.

In [11]:
SENTENCE_SPLIT_REGEX = re.compile(r'[.!?\n]+')

def read_docx(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f'Arquivo DOCX não encontrado: {path}')
    document = Document(path)
    return '\n'.join(paragraph.text for paragraph in document.paragraphs)

def read_pdf(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f'Arquivo PDF não encontrado: {path}')
    text_parts = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text_parts.append(page.extract_text() or '')
    return '\n'.join(text_parts)

def text_to_sentences(text: str) -> List[List[str]]:
    sentences = []
    for raw in SENTENCE_SPLIT_REGEX.split(text):
        tokens = simple_preprocess(raw, deacc=True, min_len=3)
        if tokens:
            sentences.append(tokens)
    return sentences

def build_corpus(texts: List[str]) -> List[List[str]]:
    corpus = []
    for text in texts:
        corpus.extend(text_to_sentences(text))
    if not corpus:
        raise ValueError('Nenhuma sentença válida encontrada para treinar o Word2Vec.')
    return corpus

def train_word2vec(sentences: List[List[str]], vector_size: int, epochs: int) -> Word2Vec:
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=5,
        min_count=1,
        workers=os.cpu_count() or 1,
        sg=1,  # Skip-gram
        epochs=epochs
    )
    return model

## 1. Carregamento e preparação dos documentos

In [14]:
print("Lendo o arquivo: ", DOCX_FILE)
docx_text = read_docx(DOCX_FILE)
print("Lendo o arquivo: ", PDF_FILE)
pdf_text = read_pdf(PDF_FILE)
print(f'DOCX: {len(docx_text.split())} palavras | PDF: {len(pdf_text.split())} palavras')
corpus = build_corpus([docx_text, pdf_text])
print(f'Sentenças para treino: {len(corpus)}')

Lendo o arquivo:  C:\Users\anton\OneDrive\Documentos\Projetos\Fatec\Comparador de Ementas\Projeto Pedagógico do Curso Superior de Tecnologia em ADS - Fatec Zona Leste 2022.10.10.docx
Lendo o arquivo:  C:\Users\anton\OneDrive\Documentos\Projetos\Fatec\Comparador de Ementas\PPC_CST Análise e Desenvolvimento de Sistemas.pdf
DOCX: 15848 palavras | PDF: 29726 palavras
Sentenças para treino: 5982


## 2. Treinamento do modelo Word2Vec (Skip-gram)

In [15]:
w2v_model = train_word2vec(corpus, vector_size=WORD2VEC_VECTOR_SIZE, epochs=WORD2VEC_EPOCHS)
word_vectors = w2v_model.wv
print(f'Modelo treinado com {len(word_vectors.index_to_key)} termos únicos.')

Modelo treinado com 3553 termos únicos.


## 3. Leitura dos CSVs
Cada CSV deve conter ao menos a coluna especificada em COLUMN_DF_A e COLUMN_DF_B.

In [16]:
df_a = pd.read_csv(CSV_A_PATH, encoding="utf-8", header=None, names=["title", "syllabus", "more_info"])
df_b = pd.read_csv(CSV_B_PATH, encoding="utf-8", header=None, names=["title", "syllabus", "more_info"])
display(df_a.head())
display(df_b.head())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe7 in position 10: invalid continuation byte

## 4. Funções para embeddings e similaridade

In [ ]:
from numpy.linalg import norm

def text_embedding(text: Optional[str]) -> Optional[np.ndarray]:
    if not isinstance(text, str) or not text.strip():
        return None
    tokens = simple_preprocess(text, deacc=True, min_len=3)
    vectors = [word_vectors[token] for token in tokens if token in word_vectors]
    if not vectors:
        return None
    return np.mean(vectors, axis=0)

def cosine_similarity(vec_a: Optional[np.ndarray], vec_b: Optional[np.ndarray]) -> float:
    if vec_a is None or vec_b is None:
        return 0.0
    denom = norm(vec_a) * norm(vec_b)
    if denom == 0:
        return 0.0
    score = float(np.dot(vec_a, vec_b) / denom)
    return max(0.0, min(1.0, score))

## 5. Comparação cruzada entre os dataframes
Calcula a similaridade de cada texto de df_a[COLUMN_DF_A] contra todos de df_b[COLUMN_DF_B].

In [ ]:
records = []
for idx_a, row_a in df_a.iterrows():
    emb_a = text_embedding(row_a.get(COLUMN_DF_A))
    for idx_b, row_b in df_b.iterrows():
        emb_b = text_embedding(row_b.get(COLUMN_DF_B))
        score = round(cosine_similarity(emb_a, emb_b), 4)
        records.append({
            'df_a_index': idx_a,
            'df_a_text': row_a.get(COLUMN_DF_A),
            'df_b_index': idx_b,
            'df_b_text': row_b.get(COLUMN_DF_B),
            'similarity': score
        })
comparison_df = pd.DataFrame(records)
comparison_df.head()

### Matriz de similaridade
Visualização estilo matriz (linhas = df_a, colunas = df_b).

In [ ]:
similarity_matrix = comparison_df.pivot(index='df_a_index', columns='df_b_index', values='similarity')
similarity_matrix